In [1]:
import optuna
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas as pd
import datetime
import io
import contextlib
import json



In [2]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available: 2


In [3]:
NUM_CLASSES = 100
INPUT_SHAPE = (32, 32, 3)
MAX_EPOCHS = 50
VALIDATION_SPLIT = 0.2

In [4]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data(label_mode='fine')

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
# Global list to store results
trial_log = []


def objective(trial):
    K.clear_session()

    lr = trial.suggest_categorical("lr", [1e-5, 1e-4, 1e-3])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])
    dropout_rate = trial.suggest_categorical("dropout", [0.2, 0.3, 0.4])

    # Filters
    filters_1 = trial.suggest_categorical("filters_1", [64, 128, 256])  # always included
    filters_2 = trial.suggest_categorical("filters_2", [0, 64, 128, 256])
    filters_3 = trial.suggest_categorical("filters_3", [0, 64, 128, 256])

    dense_units = trial.suggest_categorical("dense_units", [128, 256, 512])
    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    optimizer = Adam(learning_rate=lr) if optimizer_name == "Adam" else RMSprop(learning_rate=lr)
    
    print(f"\nTrial {trial.number} Hyperparameters:")
    print(f"  Learning rate: {lr}")
    print(f"  Optimizer:     {optimizer_name}")
    print(f"  Dropout:       {dropout_rate}")
    print(f"  Filters:       [{filters_1}, {filters_2}, {filters_3}]")
    print(f"  Dense units:   {dense_units}")
    print(f"  Batch size:    {batch_size}")


    model = Sequential()

    # Conv Block 1 (always included)
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu', input_shape=INPUT_SHAPE))
    model.add(BatchNormalization())
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(dropout_rate))

    # Conv Block 2 (optional)
    if filters_2 > 0:
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Conv Block 3 (optional)
    if filters_3 > 0:
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Dense block
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

    # Train the model
    history = model.fit(
        x_train, y_train_cat,
        validation_split=VALIDATION_SPLIT,
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        verbose=1,
        callbacks=[early_stopping, reduce_lr]
    )

    val_acc = history.history['val_accuracy'][-1]
    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    
    with io.StringIO() as buf, contextlib.redirect_stdout(buf):
        model.summary()
        model_summary_str = buf.getvalue()


    trial_log.append({
        'trial': trial.number,
        'lr': lr,
        'optimizer': optimizer_name,
        'dropout': dropout_rate,
        'filters_1': filters_1,
        'filters_2': filters_2,
        'filters_3': filters_3,
        'dense_units': dense_units,
        'batch_size': batch_size,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'epochs_ran': len(history.history['accuracy']),
        'timestamp': datetime.datetime.now().isoformat(),
        'model_summary': model_summary_str 
    })

    return val_acc


In [ ]:
try:
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=25, show_progress_bar=True)

except KeyboardInterrupt:
    print("\nOptimization interrupted. Saving progress...")

finally:
    # Save to CSV and JSON no matter what
    pd.DataFrame(trial_log).to_csv("optuna_trials.csv", index=False)

    with open("optuna_trials_with_summaries.json", "w") as f:
        json.dump(trial_log, f, indent=2)

    print("Trials saved to 'optuna_trials.csv' and 'optuna_trials_with_summaries.json'")


[I 2025-03-27 05:14:59,821] A new study created in memory with name: no-name-d8147ea2-1a8a-47ae-832e-70d356dd75fb


  0%|          | 0/30 [00:00<?, ?it/s]


Trial 0 Hyperparameters:
  Learning rate: 1e-05
  Optimizer:     RMSprop
  Dropout:       0.4
  Filters:       [256, 64, 64]
  Dense units:   512
  Batch size:    128


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 52s 120ms/step - accuracy: 0.0089 - loss: 5.8156 - val_accuracy: 0.0093 - val_loss: 4.6240 - learning_rate: 1.0000e-05
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 81ms/step - accuracy: 0.0112 - loss: 5.0278 - val_accuracy: 0.0130 - val_loss: 4.5915 - learning_rate: 1.0000e-05
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - accuracy: 0.0136 - loss: 4.7966 - val_accuracy: 0.0215 - val_loss: 4.5768 - learning_rate: 1.0000e-05
Epoch 4/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 87ms/step - accuracy: 0.0145 - loss: 4.6811 - val_accuracy: 0.0253 - val_loss: 4.5674 - learning_rate: 1.0000e-05
Epoch 5/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 87ms/step - accuracy: 0.0165 - loss: 4.6334 - val_accuracy: 0.0316 - val_loss: 4.5546 - learning_rate: 1.0000e-05
Epoch 6/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step - accuracy: 0.0178 - loss: 4.5943 - val_accuracy: 0.0341 - val_loss: 4.5311 - learning_rate: 1.0000e-05
Epoch 7/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 87ms/st

...

 Total params: 10,217,674 (38.98 MB)

 Trainable params: 5,108,068 (19.49 MB)

 Non-trainable params: 1,536 (6.00 KB)

 Optimizer params: 5,108,070 (19.49 MB)

[I 2025-03-27 10:44:04,602] Trial 20 finished with value: 0.5931000113487244 and parameters: {'lr': 0.0001, 'optimizer': 'RMSprop', 'dropout': 0.2, 'filters_1': 256, 'filters_2': 256, 'filters_3': 256, 'dense_units': 512, 'batch_size': 64}. Best is trial 20 with value: 0.5931000113487244.

Trial 21 Hyperparameters:
  Learning rate: 0.0001
  Optimizer:     RMSprop
  Dropout:       0.2
  Filters:       [256, 256, 256]
  Dense units:   512
  Batch size:    64
Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 67ms/step - accuracy: 0.0594 - loss: 4.3901 - val_accuracy: 0.1116 - val_loss: 3.8274 - learning_rate: 1.0000e-04
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 65ms/step - accuracy: 0.1682 - loss: 3.5330 - val_accuracy: 0.2483 - val_loss: 3.1100 - learning_rate: 1.0000e-04
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 65ms/step - accuracy: 0.2483 - loss: 3.0740 - val_accuracy: 0.2953 - val_loss: 2.8445 - learning_rate: 1.0000e-04
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 65ms/step - accuracy:

In [ ]:
pd.DataFrame(trial_log).to_csv("optuna_trials.csv", index=False)

with open("optuna_trials_with_summaries.json", "w") as f:
    json.dump(trial_log, f, indent=2)

print("Trials saved to 'optuna_trials.csv' and 'optuna_trials_with_summaries.json'")